# Compare pilot 01 against the frozen baseline

Run after the 60-step pilot has saved its adapter. Upload these complete folders to the same remote Colab runtime: `prepared_clinical_dataset/`, the numbered baseline result folder, and `pilot_01/` (including `adapter/`). Configure their paths below. This notebook runs **10 adapter inference cases only**; it reuses the original saved baseline outputs rather than rerunning baseline generation.

It verifies the dataset, selection, prompt hashes, training-data hash, base-model revision, and copied comparison contract before loading the adapter. Decoding uses the contract's FP32, greedy decoding, and 2,048-token budget. The original baseline metric definitions and generation loop are reused unchanged. References never enter the model prompt.

It saves adapter generations/raw tokens, before/after aggregate metrics, and a per-case comparison CSV containing both outputs. Adapter file hashes prevent cached results being reused for changed weights. A stopped run resumes completed cases. No training or paid judge calls are performed.

Use the same baseline software versions; setup stops if Torch or Transformers differs. Exact value matching is not semantic or clinical validation. These 10 reused validation cases form a development comparison set.


In [1]:
%pip install --progress-bar on "transformers==4.57.6" "datasets>=3,<5" huggingface_hub pandas accelerate "peft==0.18.1"


In [2]:
import os, json, hashlib, random, re, time, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
from huggingface_hub import HfApi, get_token
from transformers import AutoTokenizer

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        print("Enter your Hugging Face token in the VS Code input prompt.", flush=True)
        token = getpass("Hugging Face read token: ").strip()
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token


import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM
from peft import PeftModel

PREPARED_ROOT = Path("prepared_clinical_dataset")
BASELINE_DIR = Path("baseline_evaluation_diagnostic_10/900825a06d38a006_353c8393d011")
PILOT_DIR = Path("pilot_02")  # Contains adapter/, run_config.json, comparison_contract.json.
OUTPUT_ROOT = Path("pilot_02_comparison")

def runtime_path(path, sentinel):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates += [parent/path for parent in Path.cwd().parents]
        candidates.append(Path("/content")/path)
    for candidate in candidates:
        if (candidate/sentinel).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Kernel directory: {Path.cwd()}; missing {sentinel} under {path}. "
                            "Upload the complete folder to this runtime and correct the configured path.")

def file_sha(path):
    sha = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""):
            sha.update(block)
    return sha.hexdigest()

print("Kernel working directory:", Path.cwd(), flush=True)
PREPARED_ROOT = runtime_path(PREPARED_ROOT, "latest.json")
BASELINE_DIR = runtime_path(BASELINE_DIR, "comparison_contract.json")
PILOT_DIR = runtime_path(PILOT_DIR, "adapter/adapter_config.json")
contract = json.loads((BASELINE_DIR/"comparison_contract.json").read_text(encoding="utf-8"))
assert json.loads((PILOT_DIR/"comparison_contract.json").read_text(encoding="utf-8")) == contract
training_config = json.loads((PILOT_DIR/"run_config.json").read_text(encoding="utf-8"))
assert training_config["run_training"] is True
assert training_config["comparison_contract_sha256"] == digest(contract)
version = contract["dataset_version"]
if not isinstance(version, str) or Path(version).name != version or version in (".", ".."):
    raise ValueError("Invalid dataset version")
prepared_dir = PREPARED_ROOT/version
manifest = json.loads((prepared_dir/"manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir/"dataset"))
for split, sha in manifest["split_sha256"].items():
    assert digest(prepared[split].to_list()) == sha, f"Prepared {split} content changed"
assert training_config["train_sha256"] == manifest["split_sha256"]["train"]
assert manifest["split_sha256"]["validation"] == contract["validation_sha256"]
assert manifest["config"]["model"] == contract["model"]
assert manifest["config"]["model_revision"] == contract["base_model_revision"]
cases = prepared["validation"].to_list()
assert [c["case_id"] for c in cases] == contract["case_ids"]
assert digest([c["prompt_text"] for c in cases]) == contract["prompt_sha256"]
selection = json.loads((BASELINE_DIR/"selection.json").read_text(encoding="utf-8"))
assert len(selection) == len(cases)
for saved, case in zip(selection, cases):
    assert all(saved[key] == case[key] for key in saved), "Baseline selection changed"
baseline_generations = json.loads((BASELINE_DIR/"generations.json").read_text(encoding="utf-8"))
baseline_diagnostics = json.loads((BASELINE_DIR/"generation_diagnostics.json").read_text(encoding="utf-8"))
assert all(c["case_id"] in baseline_generations and c["case_id"] in baseline_diagnostics for c in cases)

# Refuse a silent change in generation software; restore the baseline runtime if this fails.
for package in ("transformers", "torch"):
    actual = torch.__version__ if package == "torch" else importlib.metadata.version(package)
    assert actual == contract[package], f"{package} differs from baseline: {actual} vs {contract[package]}"
assert contract["do_sample"] is False and contract["tokenizer_add_special_tokens"] is False
assert contract["metrics_version"] == 1
MAX_NEW_TOKENS, SEED = contract["max_new_tokens"], contract["seed"]
MODEL_ID, model_revision = contract["model"], contract["base_model_revision"]
INFERENCE_DTYPE = contract["dtype"]
device = "cuda" if torch.cuda.is_available() else "cpu"
if INFERENCE_DTYPE == "float32":
    dtype = torch.float32
elif INFERENCE_DTYPE == "bfloat16":
    if device != "cuda" or not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError("Native BF16 is required by this baseline contract")
    dtype = torch.bfloat16
else:
    raise ValueError("Unsupported contract precision")
adapter_dir = PILOT_DIR/"adapter"
adapter_config = json.loads((adapter_dir/"adapter_config.json").read_text(encoding="utf-8"))
assert adapter_config["base_model_name_or_path"] == MODEL_ID
adapter_files = sorted(adapter_dir.glob("adapter_model*.safetensors"))
assert adapter_files, "Missing adapter weights"
evaluation_config = {"comparison_contract_sha256": digest(contract),
    "adapter_file_sha256": {p.name: file_sha(p) for p in adapter_files + [adapter_dir/"adapter_config.json"]},
    "baseline_outputs_sha256": digest(baseline_generations),
    "baseline_diagnostics_sha256": digest(baseline_diagnostics),
    "metric_source_sha256": "021d67e1183e4c5be0f1af69cedcbb73749e0388fc27033b305a25c86640790e",
    "peft": importlib.metadata.version("peft")}
output_dir = OUTPUT_ROOT/(version+"_"+digest(evaluation_config)[:12])
output_dir.mkdir(parents=True, exist_ok=True)
config_path = output_dir/"evaluation_config.json"
if config_path.exists():
    assert json.loads(config_path.read_text(encoding="utf-8")) == evaluation_config
else:
    config_path.write_text(json.dumps(evaluation_config, indent=2), encoding="utf-8")
(output_dir/"comparison_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")
(output_dir/"selection.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")
print("Verified frozen selection:", len(cases), "cases; adapter:", adapter_dir, flush=True)
token = read_token()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=token)
random.seed(SEED)
torch.manual_seed(SEED)
print("Loading base model plus pilot adapter on", device, dtype, flush=True)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=model_revision,
                                                token=token, dtype=dtype).to(device)
model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=False).eval()


Kernel working directory: /content
Verified frozen selection: 10 cases; adapter: /content/pilot_02/adapter
Enter your Hugging Face token in the VS Code input prompt.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading base model plus pilot adapter on cuda torch.float32


In [3]:
generations_path = output_dir / "generations.json"
diagnostics_path = output_dir / "generation_diagnostics.json"
generations = json.loads(generations_path.read_text()) if generations_path.exists() else {}
diagnostics = json.loads(diagnostics_path.read_text()) if diagnostics_path.exists() else {}
started = time.monotonic()
for case in cases:
    key = case["case_id"]
    if generations.get(key, "").strip() and key in diagnostics:
        continue
    inputs = tokenizer(case["prompt_text"], return_tensors="pt", add_special_tokens=False).to(device)
    n_prompt = inputs.input_ids.shape[1]
    if n_prompt + MAX_NEW_TOKENS > model.config.max_position_embeddings:
        raise ValueError(f"{key}: prompt plus generation budget exceeds model context")
    with torch.inference_mode():
        # Keep only the final position to avoid materializing all prompt logits.
        logits = model(**inputs, logits_to_keep=1).logits[:, -1, :]
        finite = bool(torch.isfinite(logits).all().item())
        del logits
        if finite:
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
            ids = generated[0, n_prompt:].tolist()
        else:
            ids = []
    text = tokenizer.decode(ids, skip_special_tokens=True)
    eos = model.generation_config.eos_token_id
    eos = eos if isinstance(eos, list) else [eos]
    diagnostics[key] = {"prompt_tokens": n_prompt, "generated_tokens": len(ids),
        "finite_first_token_logits": finite, "empty_output": not text.strip(),
        "reached_budget": len(ids) >= MAX_NEW_TOKENS,
        "ended_with_eos": bool(ids) and ids[-1] in eos,
        "token_ids": ids, "raw_decoded": tokenizer.decode(ids, skip_special_tokens=False)}
    diagnostics_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
    if not finite or not text.strip():
        raise RuntimeError(f"{key}: failed generation. Diagnostics saved; rerun retries this case.")
    generations[key] = text
    generations_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{len(generations)}/{len(cases)}: {key}, {len(ids)} tokens, {time.monotonic()-started:.1f}s; {text[:100]!r}", flush=True)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1/10: validation:2761, 481 tokens, 25.7s; '{"visit motivation":"Large tender erythematous morpheaform plaque without ulceration","admission":[{'
2/10: validation:465, 2048 tokens, 126.9s; '{"visit motivation":"Recurrent low-grade bladder cancer","admission":[{"reason":"None","date":"None"'
3/10: validation:105, 1214 tokens, 187.4s; '{"visit motivation":"Pain in the right eye","admission":[{"reason":"Pain in the right eye","date":"N'
4/10: validation:1161, 636 tokens, 218.5s; '{"visit motivation":"Repeated cough and sputum for one year and aggravation with chest tightness","a'
5/10: validation:1030, 2048 tokens, 319.8s; '{"visit motivation":"Acute pain of the left mandible","admission":[{"reason":"Acute pain of the left'
6/10: validation:935, 485 tokens, 343.7s; '{"visit motivation":"Recurrent bouts of abdominal pain and fluctuating fevers","admission":[{"reason'
7/10: validation:583, 719 tokens, 379.6s; '{"visit motivation":"Gross hematuria","admission":[{"reason":"Slow-growing right r

In [4]:
TOP_LEVEL_KEYS = (
    "visit motivation", "admission", "patient information", "patient medical history",
    "surgeries", "symptoms", "medical examinations", "diagnosis tests", "treatments", "discharge")
FORMAT_INSTRUCTION = (
    "Extract the clinical facts stated in the note into one JSON object. "
    "Return JSON only, without markdown or commentary. Use these top-level fields: "
    + ", ".join(TOP_LEVEL_KEYS) + ". "
    "Use the string None only for information absent from the note. "
    "Do not invent facts or repeat identical entries.")

def no_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Nonstandard JSON constant: " + value)

def load_object(text, strict=True):
    obj = json.loads(text, strict=strict, object_pairs_hook=no_duplicate_keys,
                     parse_constant=reject_constant)
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")
    return obj

def clean_reference(raw):
    text = raw.strip()
    fenced = text.startswith("```") and text.endswith("```")
    if fenced:
        text = re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    try:
        obj = load_object(text)
        repair = "removed_markdown_fence" if fenced else "none"
    except ValueError:
        # Only unescaped control characters are tolerated; serialization escapes them.
        # No guessed commas, missing braces, invented values, or LLM repair.
        obj = load_object(text, strict=False)
        repair = "escaped_control_characters" + ("_and_removed_fence" if fenced else "")
    if set(obj) != set(TOP_LEVEL_KEYS):
        raise ValueError("Unexpected top-level schema")
    canonical = json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    assert load_object(canonical) == obj
    return canonical, repair

def strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        return re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    return text

def facts(value, path=""):
    """Multiset of (field path, normalized value); array order is ignored."""
    result = Counter()
    if isinstance(value, dict):
        for key, child in value.items():
            result.update(facts(child, f"{path}.{key}" if path else key))
    elif isinstance(value, list):
        for child in value:
            result.update(facts(child, path))
    elif value is not None:
        normalized = " ".join(str(value).casefold().split())
        if normalized not in ("", "none", "null", "unknown", "not provided"):
            result[(path, normalized)] += 1
    return result

def key_paths(value, prefix=""):
    result = set()
    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else key
            result.add(path)
            result.update(key_paths(child, path))
    elif isinstance(value, list):
        for child in value:
            result.update(key_paths(child, prefix))
    return result

def evaluate_output(text, reference):
    expected = load_object(reference)
    raw_valid = True
    try:
        load_object(text)
    except ValueError:
        raw_valid = False
    try:
        actual = load_object(strip_fences(text))
        error = None
    except ValueError as exc:
        actual, error = None, str(exc)
    expected_facts = facts(expected)
    actual_facts = facts(actual)
    matches = sum((expected_facts & actual_facts).values())
    n_actual, n_expected = sum(actual_facts.values()), sum(expected_facts.values())
    precision = matches / n_actual if n_actual else 0.0
    recall = matches / n_expected if n_expected else 0.0
    return {"empty": not text.strip(), "raw_json_object_valid": raw_valid,
            "json_object_valid_after_fence_removal": actual is not None,
            "parse_error": error, "exact_key_match": actual is not None and key_paths(actual) == key_paths(expected),
            "nonmissing_generated_values": n_actual, "reference_values": n_expected,
            "matched_values": matches, "placeholder_only": actual is not None and n_actual == 0,
            "value_precision": precision, "value_recall": recall,
            "value_f1": 2*precision*recall/(precision+recall) if precision+recall else 0.0}

ref = '{"age":"29", "symptoms":[{"name":"pain"}]}'
assert evaluate_output('```json\n'+ref+'\n```', ref)["value_f1"] == 1
assert not evaluate_output('```json\n'+ref+'\n```', ref)["raw_json_object_valid"]
assert evaluate_output('{"age":"None"}', ref)["placeholder_only"]
assert evaluate_output('', ref)["value_recall"] == 0
assert evaluate_output('{"age":"29","age":"30"}', ref)["parse_error"]
assert evaluate_output('{"age":"29", "symptoms":[{"name":"pain"},{"name":"pain"}]}', ref)["value_precision"] < 1
print("Metric self-checks passed.")


Metric self-checks passed.


In [5]:
assert all(c["case_id"] in generations for c in cases), "Complete adapter generation first"

def score_run(outputs, diagnostics):
    records = []
    for case in cases:
        key = case["case_id"]
        records.append({"case_id": key, **evaluate_output(outputs[key], case["reference"]),
            "reached_budget": bool(diagnostics[key].get("reached_budget")),
            "generated_tokens": diagnostics[key]["generated_tokens"], "generated": outputs[key],
            "reference": case["reference"]})
    frame = pd.DataFrame(records)
    summary = {"selected_cases": len(cases), "evaluated_cases": len(records)}
    for name in ("empty", "raw_json_object_valid", "json_object_valid_after_fence_removal",
                 "exact_key_match", "placeholder_only", "reached_budget"):
        summary[name+"_cases"] = int(frame[name].sum())
    for name in ("value_precision", "value_recall", "value_f1"):
        summary["mean_"+name] = float(frame[name].mean())
    for key,col in (("matched_values","matched_values"), ("reference_values","reference_values"),
                    ("generated_values","nonmissing_generated_values")):
        summary[key] = int(frame[col].sum())
    return frame, summary

before, baseline_summary = score_run(baseline_generations, baseline_diagnostics)
after, adapter_summary = score_run(generations, diagnostics)
before.to_csv(output_dir/"baseline_per_case.csv", index=False)
after.to_csv(output_dir/"finetuned_per_case.csv", index=False)
(output_dir/"baseline_aggregate.json").write_text(json.dumps(baseline_summary, indent=2), encoding="utf-8")
(output_dir/"finetuned_aggregate.json").write_text(json.dumps(adapter_summary, indent=2), encoding="utf-8")
comparison_rows = [{"metric":key, "baseline":baseline_summary[key], "finetuned":adapter_summary[key],
                    "change":adapter_summary[key]-baseline_summary[key]} for key in baseline_summary]
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(output_dir/"comparison.csv", index=False)
(output_dir/"comparison.json").write_text(json.dumps(comparison_rows, indent=2), encoding="utf-8")
case_comparison = before.drop(columns=["reference"]).merge(after, on="case_id", suffixes=("_baseline","_finetuned"), validate="one_to_one")
case_comparison.to_csv(output_dir/"case_comparison.csv", index=False)
display(comparison)
print("Per-case outputs and metrics:", output_dir.resolve())
print("Inspect reports against the source notes. Exact-value matching penalizes paraphrases; lower loss or higher JSON validity alone does not establish clinical correctness.")


,metric,baseline,finetuned,change
0,selected_cases,10.000000,10.000000,0.000000
1,evaluated_cases,10.000000,10.000000,0.000000
2,empty_cases,0.000000,0.000000,0.000000
3,raw_json_object_valid_cases,0.000000,5.000000,5.000000
4,json_object_valid_after_fence_removal_cases,4.000000,5.000000,1.000000
5,exact_key_match_cases,0.000000,2.000000,2.000000
6,placeholder_only_cases,0.000000,0.000000,0.000000
7,reached_budget_cases,3.000000,4.000000,1.000000
8,mean_value_precision,0.006667,0.125493,0.118826
9,mean_value_recall,0.002000,0.097183,0.095183


Per-case outputs and metrics: /content/pilot_02_comparison/900825a06d38a006_9102ae51256b
Inspect reports against the source notes. Exact-value matching penalizes paraphrases; lower loss or higher JSON validity alone does not establish clinical correctness.


In [6]:
import shutil
archive = shutil.make_archive(str(output_dir.resolve()), "zip", root_dir=output_dir.parent,
                             base_dir=output_dir.name)
print("Download baseline artifacts:", archive)
# Browser Colab only: from google.colab import files; files.download(archive)


Download baseline artifacts: /content/pilot_02_comparison/900825a06d38a006_9102ae51256b.zip
